# Publish + subscribe on a Jammi topic via the in-process broker

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/recipes/trigger_streams.ipynb)

Built from [`cookbook/recipes/trigger_streams/example.py`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/recipes/trigger_streams/example.py). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures. On that GPU the
# chapter runs at `full` scale, over the published data; set SCALE = "small" to
# run the seconds-long version over the committed fixtures instead.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook==0.49.1"], check=True)
SCALE = "full" if gpu else "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

Run with `python cookbook/recipes/trigger_streams/example.py`. Exits 0
on success.

In [ ]:
from __future__ import annotations

import tempfile

import pyarrow as pa

import jammi


def events_schema() -> pa.Schema:
    return pa.schema(
        [
            pa.field("event_id", pa.int64(), nullable=False),
            pa.field("payload", pa.string(), nullable=False),
        ]
    )


def main() -> int:
    with tempfile.TemporaryDirectory() as tmp, jammi.connect(f"file://{tmp}") as db:

        # 1. Register a topic with a typed schema.
        topic_id = db.register_topic("events.demo", schema=events_schema())
        assert topic_id, "register_topic returned empty id"

        # 2. The catalog now lists the topic for the current tenant.
        topics = db.list_topics()
        assert "events.demo" in topics, f"events.demo missing from {topics}"

        # 3. Publish one batch. The broker assigns sequential offsets per
        #    topic, starting at 0 for a fresh topic.
        batch = pa.table(
            {
                "event_id": pa.array([1, 2, 3], type=pa.int64()),
                "payload": pa.array(["alpha", "beta", "gamma"], type=pa.string()),
            },
            schema=events_schema(),
        )
        offset = db.publish_topic("events.demo", batch=batch)
        assert offset == 0, f"expected offset 0, got {offset}"

        # 4. Subscribe from offset 0 — drives the backing-table replay
        #    path. `max_batches=1` matches the published batch count so
        #    the call returns immediately without racing the live tail.
        collected = db.subscribe_collect(
            "events.demo", from_offset=0
        )
        assert collected.column("event_id").to_pylist() == [1, 2, 3]
        assert collected.column("payload").to_pylist() == ["alpha", "beta", "gamma"]

        # 5. Drop the topic and confirm it leaves the catalog.
        db.drop_topic("events.demo")
        topics = db.list_topics()
        assert "events.demo" not in topics, "events.demo persisted after drop"

        # 6. Idempotent drop — `if_exists=True` swallows the missing case.
        db.drop_topic("events.demo", if_exists=True)

        # 7. Strict drop on a missing topic raises with a useful message.
        try:
            db.drop_topic("never.registered")
        except (ValueError, RuntimeError) as exc:
            assert "never.registered" in str(exc), (
                f"drop-missing error lost topic name: {exc}"
            )
        else:
            raise AssertionError("drop-missing must raise")

    print("trigger_streams: OK")
    return 0

In [ ]:
assert main() == 0